In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder

# ====================================================================
# ♾️ PROTOCOLE : "POISSON INFINITY"
# Stratégie : Nuclear Base + Seed Averaging (3 seeds) = 30 Modèles
# ====================================================================

# 1. CHARGEMENT & FEATURE ENGINEERING
print("⏳ Chargement des données...")
train_data = pd.read_csv('conversion_data_train.csv')
test_data = pd.read_csv('conversion_data_test.csv')

def feature_engineering(df):
    df_eng = df.copy()
    # Interactions stratégiques (Sniper Features)
    df_eng['interaction_age_pages'] = df_eng['age'] * df_eng['total_pages_visited']
    df_eng['pages_per_age'] = df_eng['total_pages_visited'] / (df_eng['age'] + 0.1)
    return df_eng

# On garde le bruit et les doublons (Stratégie Monster)
X = feature_engineering(train_data.drop('converted', axis=1))
y = train_data['converted']
X_test = feature_engineering(test_data)

# 2. ENCODAGE GLOBAL
categorical_cols = ['country', 'source']
cat_indices = [X.columns.get_loc(col) for col in categorical_cols]

for col in categorical_cols:
    le = LabelEncoder()
    combined = pd.concat([X[col], X_test[col]])
    le.fit(combined)
    X[col] = le.transform(X[col])
    X_test[col] = le.transform(X_test[col])

# 3. CONFIGURATION INFINITY
n_folds = 10
SEEDS = [42, 2025, 777] # Les 3 dimensions parallèles
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))
test_preds_accumulated = np.zeros(len(X_test))
fold_scores = []

print(f"🚀 Démarrage du moteur INFINITY ({n_folds} Folds x {len(SEEDS)} Seeds)...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train_fold, y_train_fold = X.iloc[train_idx], y.iloc[train_idx]
    X_val_fold, y_val_fold = X.iloc[val_idx], y.iloc[val_idx]
    
    # Stockage temporaire pour le Seed Averaging de ce pli
    fold_val_preds_avg = np.zeros(len(val_idx))
    fold_test_preds_avg = np.zeros(len(X_test))
    
    # Boucle sur les Seeds (Averaging interne)
    for seed in SEEDS:
        model = HistGradientBoostingRegressor(
            loss='poisson',           # La clé
            learning_rate=0.05,
            max_iter=600,
            max_depth=8,
            l2_regularization=0.1,
            categorical_features=cat_indices,
            random_state=seed         # Variation de la graine
        )
        
        model.fit(X_train_fold, y_train_fold)
        
        # Accumulation
        fold_val_preds_avg += model.predict(X_val_fold)
        fold_test_preds_avg += model.predict(X_test)
    
    # Moyenne des seeds pour ce pli
    fold_val_preds_avg /= len(SEEDS)
    fold_test_preds_avg /= len(SEEDS)
    
    # Stockage OOF et Global
    oof_preds[val_idx] = fold_val_preds_avg
    test_preds_accumulated += fold_test_preds_avg
    
    # Score du pli (basé sur la moyenne des seeds)
    auc_fold = roc_auc_score(y_val_fold, fold_val_preds_avg)
    fold_scores.append(auc_fold)
    print(f"  > Fold {fold+1}/{n_folds} | ROC-AUC (Avg): {auc_fold:.5f}")

# 4. FINITION
# Moyenne finale sur les 10 folds
test_preds_final = test_preds_accumulated / n_folds

print("\n🔍 Optimisation Ultra-Fine du Seuil OOF...")
best_f1 = 0
best_threshold = 0
thresholds = np.linspace(oof_preds.min(), oof_preds.max(), 1000) # Scan haute définition

for t in thresholds:
    preds_binary = (oof_preds >= t).astype(int)
    score = f1_score(y, preds_binary)
    if score > best_f1:
        best_f1 = score
        best_threshold = t

print(f"\n💎 RÉSULTATS POISSON INFINITY :")
print(f"ROC-AUC Global : {np.mean(fold_scores):.5f}")
print(f"Meilleur F1 OOF: {best_f1:.5f}")
print(f"Seuil Optimal  : {best_threshold:.6f}")

# Génération
final_test_binary = (test_preds_final >= best_threshold).astype(int)
submission = pd.DataFrame({'converted': final_test_binary})
filename = 'submission_POISSON_INFINITY.csv'
submission.to_csv(filename, index=False)

print(f"\n✅ Fichier '{filename}' généré avec {final_test_binary.sum()} conversions.")

⏳ Chargement des données...
🚀 Démarrage du moteur INFINITY (10 Folds x 3 Seeds)...
  > Fold 1/10 | ROC-AUC (Avg): 0.98683
  > Fold 2/10 | ROC-AUC (Avg): 0.98381
  > Fold 3/10 | ROC-AUC (Avg): 0.98353
  > Fold 4/10 | ROC-AUC (Avg): 0.98654
  > Fold 5/10 | ROC-AUC (Avg): 0.98750
  > Fold 6/10 | ROC-AUC (Avg): 0.98419
  > Fold 7/10 | ROC-AUC (Avg): 0.98275
  > Fold 8/10 | ROC-AUC (Avg): 0.98482
  > Fold 9/10 | ROC-AUC (Avg): 0.98769
  > Fold 10/10 | ROC-AUC (Avg): 0.98604

🔍 Optimisation Ultra-Fine du Seuil OOF...

💎 RÉSULTATS POISSON INFINITY :
ROC-AUC Global : 0.98537
Meilleur F1 OOF: 0.76848
Seuil Optimal  : 0.372191

✅ Fichier 'submission_POISSON_INFINITY.csv' généré avec 917 conversions.
